# V3 baseline dual selection: EMA-best và Raw-best

Train V3 baseline trên GPT-3.5 Mixed một lần. Sau mỗi epoch, EMA và raw đều được đánh giá riêng trên GPT-3.5 Mixed validation; mỗi nhánh lưu checkpoint tốt nhất của chính nó. Cuối notebook, hai checkpoint được test trên HC3 0% với cùng graph.

`batch-size=64` là effective batch; `micro-batch-size=8` chỉ giới hạn số graph trong GPU tại một lần forward.

In [ ]:
!pip install -q --upgrade torch-geometric transformers spacy scikit-learn tqdm
!python -m spacy download en_core_web_sm

import torch
assert torch.cuda.is_available(), 'HÃ£y báº­t GPU trong Kaggle.'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

WORK=Path('/kaggle/working/v3_dual_ema_raw_gpt35_to_hc3_0pct')
REPO=WORK/'PRDetect-Research'
BRANCH='version3'
SEED=2026
BATCH_SIZE=64
MICRO_BATCH_SIZE=64
EPOCHS=10
TEST_SPLIT='hc3_test'
MODEL_DIR=WORK/'model'
EVAL_DIR=WORK/'evaluation'

if REPO.exists(): shutil.rmtree(REPO)
WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,
                'https://github.com/tphan1420/PRDetect-Research.git',str(REPO)],check=True)
MODEL_DIR.mkdir(exist_ok=True); EVAL_DIR.mkdir(exist_ok=True)
assert (REPO/'train_v3.py').is_file()
print({'commit':subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip(),
       'seed':SEED,'batch_size':BATCH_SIZE,'micro_batch_size':MICRO_BATCH_SIZE})


In [ ]:
# Build training/validation graphs only. Test graph is built after training.
subprocess.run([sys.executable,'building_graph.py','--file','gpt3.5_mixed_train_split','gpt3.5_mixed_val_split'],cwd=REPO,check=True)

In [ ]:
# This trainer is a self-contained fork of train_v3.py. It changes only checkpoint
# selection: raw and EMA receive independent validation scores and best epochs.
DUAL_TRAINER=REPO/'train_v3_dual_best.py'
DUAL_TRAINER.write_text('"""Train the Quang V3 Core from legacy-domain graphs.\n\nNo test files are read during training or checkpoint selection.\n"""\n\nimport argparse\nfrom train_v3 import configure_determinism\nimport copy\nimport gc\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nfrom evaluate_v3_quang import (REPO, TRAINING_DOMAINS, environment, positive_int,\n                               sha256, write_json)\n\n\ndef build_parser():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--variants", nargs="+", choices=("baseline",), default=["baseline"],\n                        help="Retained for CLI compatibility; V3 Core has one baseline variant.")\n    parser.add_argument("--seeds", type=int, nargs="+", default=[2025, 2026])\n    parser.add_argument("--graph-dir", type=Path, default=REPO / "graph_data")\n    parser.add_argument("--output-dir", type=Path, default=REPO / "result/v3_baseline/model")\n    parser.add_argument("--train-file", default="gpt3.5_mixed_train_split")\n    parser.add_argument("--val-file", default="gpt3.5_mixed_val_split")\n    parser.add_argument("--struct-mode", choices=("low", "high"), default="low")\n    parser.add_argument("--num-rel", type=int, default=0, help="0: infer from training graph only")\n    parser.add_argument("--hidden", type=positive_int, default=384)\n    parser.add_argument("--num-bases", type=positive_int, default=12)\n    parser.add_argument("--layers", type=positive_int, default=2)\n    parser.add_argument("--dropout", type=float, default=0.3)\n    parser.add_argument("--in-dropout", type=float, default=0.1)\n    parser.add_argument("--edge-drop", type=float, default=0.1)\n    parser.add_argument("--struct", type=int, choices=(0, 1), default=1)\n    parser.add_argument("--gfeat", type=int, choices=(0, 1), default=0)\n    parser.add_argument("--readout", choices=("pool3",), default="pool3")\n    parser.add_argument("--aug", choices=("none", "swap", "mask", "both"), default="swap")\n    parser.add_argument("--aug-max", type=float, default=0.35)\n    parser.add_argument("--ema", type=int, choices=(0, 1), default=1)\n    parser.add_argument("--ema-decay", type=float, default=0.998)\n    parser.add_argument("--lr", type=float, default=5e-4)\n    parser.add_argument("--wd", type=float, default=0.01)\n    parser.add_argument("--epochs", type=positive_int, default=10)\n    parser.add_argument("--patience", type=positive_int, default=10)\n    parser.add_argument("--batch-size", type=positive_int, default=32,\n                        help="Effective graphs per optimizer/EMA/scheduler step")\n    parser.add_argument("--micro-batch-size", type=positive_int, default=8,\n                        help="Graph donor pool size; use the same value in both variants")\n    parser.add_argument("--val-batch-size", "--eval-batch-size", dest="val_batch_size",\n                        type=positive_int, default=64,\n                        help="Also controls the donor pool of the fixed validation swap proxy")\n    parser.add_argument("--select", choices=("lossaug", "loss", "acc", "auc"), default="lossaug")\n    parser.add_argument("--amp", type=int, choices=(0, 1), default=0)\n    parser.add_argument("--device", choices=("cpu", "cuda"), default="cuda")\n    parser.add_argument("--deterministic", action=argparse.BooleanOptionalAction, default=True,\n                        help="Use reproducible algorithms (default: on); --no-deterministic restores legacy behavior")\n    parser.add_argument("--skip-completed", action="store_true")\n    parser.add_argument("--overwrite", action="store_true")\n    return parser\n\n\ndef validate_args(args):\n    pairs = {(item["train_file"], item["val_file"]) for item in TRAINING_DOMAINS.values()}\n    if (args.train_file, args.val_file) not in pairs:\n        raise ValueError("Use a supported train/validation pair from TRAINING_DOMAINS")\n    if len(set(args.seeds)) != len(args.seeds) or len(set(args.variants)) != len(args.variants):\n        raise ValueError("Seeds/variants must not contain duplicates")\n    for name in ("dropout", "in_dropout", "edge_drop", "aug_max"):\n        if not 0 <= getattr(args, name) <= 1:\n            raise ValueError(f"{name} must be in [0,1]")\n    for name in ("lr",):\n        if not math.isfinite(getattr(args, name)) or getattr(args, name) <= 0:\n            raise ValueError(f"{name} must be finite and positive")\n    if not math.isfinite(args.wd) or args.wd < 0 or not 0 <= args.ema_decay < 1 or args.num_rel < 0:\n        raise ValueError("Invalid weight decay, EMA decay or relation count")\n    if args.micro_batch_size > args.batch_size:\n        raise ValueError("micro-batch-size must not exceed batch-size")\n\n\ndef effective_readout(args, variant):\n    return args.readout\n\n\ndef build_model(args, variant, R):\n    from model.QuangV3 import Net\n\n    kwargs = dict(d=args.hidden, nb=args.num_bases, L=args.layers, p=args.dropout,\n                  p_in=args.in_dropout, edge_drop=args.edge_drop,\n                  use_struct=bool(args.struct), use_gfeat=bool(args.gfeat),\n                  readout=effective_readout(args, variant), aug=args.aug,\n                  aug_max=args.aug_max, mode=args.struct_mode)\n    return Net(R, **kwargs)\n\n\ndef load_graph(path, source, R, mode):\n    """Validate original cache schema and exact source label order, then keep on CPU."""\n    import numpy as np\n    import torch\n    from model import QuangV3 as v3\n\n    original_load = v3._load_pickle\n\n    def checked_load(filename):\n        raw = original_load(filename)\n        if not isinstance(raw, dict):\n            raise ValueError("Use dictionary caches from building_graph.py")\n        if any(key in raw for key in ("encoder_metadata", "schema_version", "all_cls_embeddings",\n                                      "all_quality", "quality_features")):\n            raise ValueError("Use legacy V1 graphs, not aligned/V2 caches")\n        for x, edge, rel in zip(raw["all_token_embeddings"], raw["all_edge_index"], raw["all_edge_type"]):\n            edge, rel = torch.as_tensor(edge), torch.as_tensor(rel)\n            if x.ndim != 2 or x.shape[0] == 0 or x.shape[1] != 768:\n                raise ValueError("Invalid legacy node embeddings; do not use aligned/V2 graphs")\n            if edge.shape != (2, rel.numel()) or rel.numel() == 0:\n                raise ValueError("Invalid relation/edge shapes")\n            if int(rel.min()) < 0 or int(edge.min()) < 0 or int(edge.max()) >= x.shape[0]:\n                raise ValueError("Invalid relation/edge indices")\n            if R and int(rel.max()) >= R:\n                raise ValueError("Validation relation IDs exceed training R; refusing silent clipping")\n        sizes = [len(raw[k]) for k in ("all_token_embeddings", "all_edge_index", "all_edge_type", "y")]\n        if len(set(sizes)) != 1:\n            raise ValueError("Cache arrays have different numbers of samples")\n        return raw\n\n    v3._load_pickle = checked_load\n    try:\n        ds = v3.GraphSet(str(path), R=R or None, mode=mode, store_dev="cpu")\n    finally:\n        v3._load_pickle = original_load\n    expected = []\n    with source.open(encoding="utf-8") as stream:\n        for line in stream:\n            if line.strip():\n                label = json.loads(line)["label"]\n                expected.append(int(label) if isinstance(label, (int, float)) else int("human" in str(label).lower()))\n    if not np.array_equal(ds.y.numpy(), expected) or set(expected) != {0, 1}:\n        raise ValueError("Cache/source labels or counts differ; rebuild with building_graph.py")\n    return ds\n\n\ndef validation_loss(probabilities, truth):\n    import numpy as np\n\n    p = np.clip(probabilities, 1e-6, 1 - 1e-6)\n    return float(-(truth * np.log(p) + (1 - truth) * np.log(1 - p)).mean())\n\n\ndef atomic_save(payload, path):\n    import torch\n\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(payload, temporary)\n    temporary.replace(path)\n\n\ndef signature(args, variant, seed, data_hashes):\n    config = vars(args).copy()\n    for key in ("variants", "seeds", "output_dir", "graph_dir", "skip_completed", "overwrite"):\n        config.pop(key)\n    config["effective_readout"] = effective_readout(args, variant)\n    code = {str(p.relative_to(REPO)): sha256(p) for p in (\n        Path(__file__), REPO / "train_v3.py", REPO / "model/QuangV3.py", REPO / "building_graph.py", REPO / "dep_vocab.py",\n        REPO / "dep_vocab.json",\n    )}\n    record = dict(config=config, variant=variant, seed=seed, data=data_hashes,\n                  code_sha256=code, environment=environment())\n    digest = hashlib.sha256(json.dumps(record, sort_keys=True).encode()).hexdigest()\n    return digest, record\n\n\ndef completed_run(path, record_path, run_signature):\n    if not path.is_file() or not record_path.is_file():\n        return False\n    record = json.loads(record_path.read_text(encoding="utf-8"))\n    return (record.get("complete") is True and record.get("run_signature") == run_signature\n            and record.get("checkpoint_sha256") == sha256(path))\n\n\ndef train_one(args, variant, seed, train, val, run_signature, provenance):\n    import numpy as np\n    import torch\n    import torch.nn.functional as F\n    from model.QuangV3 import predict, set_seed\n    from evaluate_v3_quang import score_probabilities\n\n    device = torch.device(args.device)\n    set_seed(seed)\n    model = build_model(args, variant, train.R).to(device)\n    teacher = copy.deepcopy(model).eval() if args.ema else None\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)\n    steps_per_epoch = math.ceil(train.G / args.batch_size)\n    total = steps_per_epoch * args.epochs\n    warm = max(1, int(total * 0.05))\n    schedule = torch.optim.lr_scheduler.LambdaLR(\n        optimizer, lambda s: (s + 1) / warm if s < warm else\n        0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))) * 0.95 + 0.05,\n    )\n    amp = bool(args.amp) and device.type == "cuda"\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n    gm = train.GF.float().mean(0).cpu()\n    gs = train.GF.float().std(0).cpu().clamp_min(1e-6)\n    ema_checkpoint_path = args.output_dir / f"v3_{variant}_ema_seed{seed}.pt"\n    raw_checkpoint_path = args.output_dir / f"v3_{variant}_raw_seed{seed}.pt"\n    record_path = args.output_dir / f"v3_{variant}_dual_seed{seed}.train.json"\n    saved_args = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}\n    saved_args.update(seeds=[seed], cons=0.0, readout=effective_readout(args, variant))\n    history = []\n    best_ema_score = best_raw_score = float("inf")\n    best_ema_epoch = best_raw_epoch = 0\n    bad = step = 0\n    start = time.time()\n    sampler_generator = torch.Generator(device="cpu").manual_seed(seed)\n    print(f"{variant} seed={seed} | R={train.R} mode={args.struct_mode} | "\n          f"effective batch={args.batch_size}, micro={args.micro_batch_size}", flush=True)\n\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        permutation = torch.randperm(train.G, generator=sampler_generator)\n        epoch_loss = 0.0\n        for offset in range(0, train.G, args.batch_size):\n            indices = permutation[offset:offset + args.batch_size]\n            optimizer.zero_grad(set_to_none=True)\n            for micro in indices.split(args.micro_batch_size):\n                batch = train.batch(micro, device, gm, gs)\n                weight = batch["B"] / indices.numel()\n                branch_weight = 0.5 if args.aug != "none" else 1.0\n                # Release clean activations BEFORE the augmented forward. The\n                # effective step still averages all examples/branches exactly.\n                with torch.autocast(device.type, dtype=torch.float16, enabled=amp):\n                    clean = model(batch)\n                    clean_loss = F.binary_cross_entropy_with_logits(clean.float(), batch["y"])\n                scaler.scale(weight * branch_weight * clean_loss).backward()\n                epoch_loss += branch_weight * float(clean_loss.detach()) * batch["B"]\n                del clean, clean_loss\n                if args.aug != "none":\n                    with torch.autocast(device.type, dtype=torch.float16, enabled=amp):\n                        noisy = model(batch, aug_mode=args.aug)\n                        noisy_loss = F.binary_cross_entropy_with_logits(noisy.float(), batch["y"])\n                    scaler.scale(weight * 0.5 * noisy_loss).backward()\n                    epoch_loss += 0.5 * float(noisy_loss.detach()) * batch["B"]\n                    del noisy, noisy_loss\n                del batch\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            previous_scale = scaler.get_scale()\n            scaler.step(optimizer)\n            scaler.update()\n            if scaler.get_scale() >= previous_scale:\n                schedule.step()\n                step += 1\n                if teacher is not None:\n                    decay = min(args.ema_decay, (1 + step) / (10 + step))\n                    with torch.no_grad():\n                        for target, current in zip(teacher.parameters(), model.parameters()):\n                            target.mul_(decay).add_(current.detach(), alpha=1 - decay)\n        # Each weight family chooses its own best validation epoch.  The raw\n        # model and EMA teacher therefore receive identical training updates,\n        # but neither is selected using the other model\'s validation scores.\n        ema_model = teacher if teacher is not None else model\n        raw_model = model\n        truth = val.y.numpy().astype(int)\n\n        def validate(candidate):\n            probabilities = predict(candidate, val, device, gm, gs, bs=args.val_batch_size, amp=amp)\n            swapped = predict(candidate, val, device, gm, gs, bs=args.val_batch_size, amp=amp, aug_mode="swap")\n            metrics = score_probabilities(truth, probabilities, 0.5)\n            clean_loss = validation_loss(probabilities, truth)\n            swap_loss = validation_loss(swapped, truth)\n            score = {"lossaug": clean_loss + swap_loss, "loss": clean_loss,\n                     "acc": -metrics["accuracy"], "auc": -metrics["auc"]}[args.select]\n            thresholds = np.linspace(0.05, 0.95, 91)\n            accuracies = np.array([((probabilities >= threshold) == truth).mean() for threshold in thresholds])\n            threshold = float(np.median(thresholds[accuracies >= accuracies.max() - 1e-9]))\n            return probabilities, clean_loss, swap_loss, score, metrics, threshold\n\n        ema_p, ema_clean, ema_swap, ema_score, ema_metrics, ema_threshold = validate(ema_model)\n        raw_p, raw_clean, raw_swap, raw_score, raw_metrics, raw_threshold = validate(raw_model)\n        if (not math.isfinite(epoch_loss) or not math.isfinite(ema_score)\n                or not math.isfinite(raw_score)):\n            raise FloatingPointError("Non-finite loss; no completed checkpoint will be marked")\n\n        ema_improved = ema_score < best_ema_score - 1e-6\n        raw_improved = raw_score < best_raw_score - 1e-6\n        history.append(dict(\n            epoch=epoch, train_loss=epoch_loss / train.G,\n            ema_val_loss=ema_clean, ema_val_swap_loss=ema_swap,\n            ema_selection_score=ema_score, ema_metrics=ema_metrics,\n            raw_val_loss=raw_clean, raw_val_swap_loss=raw_swap,\n            raw_selection_score=raw_score, raw_metrics=raw_metrics,\n            ema_improved=ema_improved, raw_improved=raw_improved,\n        ))\n\n        # Clone both states before saving.  The counterpart state is retained\n        # only as provenance; `state` is exactly the weight family being tested.\n        current_raw = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        current_ema = {k: v.detach().cpu().clone() for k, v in ema_model.state_dict().items()}\n        common = dict(args=saved_args, R=train.R, gm=gm, gs=gs,\n                      method=f"v3_{variant}", struct_mode=args.struct_mode,\n                      training_complete=False, run_signature=run_signature,\n                      provenance=provenance, dual_selection=True)\n        if ema_improved:\n            best_ema_score, best_ema_epoch = ema_score, epoch\n            atomic_save(dict(\n                state=current_ema, raw_state=current_raw, ema_state=current_ema,\n                thr=ema_threshold, best_epoch=epoch, weight_source="ema",\n                selection_model="ema", selection_score=ema_score, **common,\n            ), ema_checkpoint_path)\n        if raw_improved:\n            best_raw_score, best_raw_epoch = raw_score, epoch\n            atomic_save(dict(\n                state=current_raw, raw_state=current_raw, ema_state=current_ema,\n                thr=raw_threshold, best_epoch=epoch, weight_source="raw",\n                selection_model="raw", selection_score=raw_score, **common,\n            ), raw_checkpoint_path)\n        bad = 0 if (ema_improved or raw_improved) else bad + 1\n        write_json(record_path, dict(\n            complete=False, run_signature=run_signature, provenance=provenance,\n            best_ema_epoch=best_ema_epoch, best_raw_epoch=best_raw_epoch,\n            best_ema_score=best_ema_score, best_raw_score=best_raw_score,\n            history=history,\n        ))\n        print(f"Epoch {epoch:02d} | loss {epoch_loss/train.G:.4f} | "\n              f"EMA val {ema_clean:.4f} ({\'best\' if ema_improved else \'keep\'}) | "\n              f"Raw val {raw_clean:.4f} ({\'best\' if raw_improved else \'keep\'})", flush=True)\n        if bad >= args.patience:\n            break\n\n    for checkpoint_path in (ema_checkpoint_path, raw_checkpoint_path):\n        finished = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n        finished["training_complete"] = True\n        atomic_save(finished, checkpoint_path)\n    write_json(record_path, dict(\n        complete=True, run_signature=run_signature, provenance=provenance,\n        checkpoint_sha256={"ema": sha256(ema_checkpoint_path), "raw": sha256(raw_checkpoint_path)},\n        best_ema_epoch=best_ema_epoch, best_raw_epoch=best_raw_epoch,\n        best_ema_score=best_ema_score, best_raw_score=best_raw_score,\n        selection=args.select, history=history, train_seconds=time.time() - start,\n    ))\n    print(f"Saved EMA: {ema_checkpoint_path}", flush=True)\n    print(f"Saved Raw: {raw_checkpoint_path}", flush=True)\n    del model, teacher, optimizer, scaler, schedule\n    gc.collect()\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n\n\ndef main(argv=None):\n    args = build_parser().parse_args(argv)\n    validate_args(args)\n    args.determinism = configure_determinism(args.deterministic)\n    import torch\n\n    if args.device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("Enable a GPU, or explicitly use --device cpu")\n    args.graph_dir, args.output_dir = args.graph_dir.resolve(), args.output_dir.resolve()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    domain = next(item for item in TRAINING_DOMAINS.values()\n                  if (item["train_file"], item["val_file"]) == (args.train_file, args.val_file))\n    sources = [REPO / domain["train_source"], REPO / domain["val_source"]]\n    graphs = [args.graph_dir / f"{name}.pkl" for name in (args.train_file, args.val_file)]\n    data_hashes = {str(p): sha256(p) for p in sources + graphs}\n    pending = []\n    for variant in args.variants:\n        for seed in args.seeds:\n            path = args.output_dir / f"v3_{variant}_ema_seed{seed}.pt"\n            record_path = args.output_dir / f"v3_{variant}_dual_seed{seed}.train.json"\n            sig, provenance = signature(args, variant, seed, data_hashes)\n            if args.skip_completed and completed_run(path, record_path, sig):\n                print(f"Skip completed: {path.name}", flush=True)\n            elif (path.exists() or record_path.exists()) and not args.overwrite:\n                raise FileExistsError(f"Incomplete/different run: {path}. Use a new output directory "\n                                      "or --overwrite; a saved best epoch is not proof training finished.")\n            else:\n                pending.append((variant, seed, sig, provenance))\n    if not pending:\n        return\n    train = load_graph(graphs[0], sources[0], args.num_rel, args.struct_mode)\n    val = load_graph(graphs[1], sources[1], train.R, args.struct_mode)\n    for variant, seed, sig, provenance in pending:\n        train_one(args, variant, seed, train, val, sig, provenance)\n\n\nif __name__ == "__main__":\n    main()\n',encoding='utf-8')
compile(DUAL_TRAINER.read_text(encoding='utf-8'),str(DUAL_TRAINER),'exec')
print('Wrote:',DUAL_TRAINER)


In [ ]:
subprocess.run(
    [sys.executable,'train_v3_dual_best.py','--variants','baseline','--seeds',str(SEED),
     '--graph-dir',str(REPO/'graph_data'),'--output-dir',str(MODEL_DIR),
     '--train-file','gpt3.5_mixed_train_split','--val-file','gpt3.5_mixed_val_split','--struct-mode','low',
     '--batch-size',str(BATCH_SIZE),'--micro-batch-size',str(MICRO_BATCH_SIZE),
     '--val-batch-size','64','--epochs',str(EPOCHS),'--patience','10','--device','cuda'],
    cwd=REPO,check=True)
EMA_CHECKPOINT=MODEL_DIR/f'v3_baseline_ema_seed{SEED}.pt'
RAW_CHECKPOINT=MODEL_DIR/f'v3_baseline_raw_seed{SEED}.pt'
TRAIN_METADATA=MODEL_DIR/f'v3_baseline_dual_seed{SEED}.train.json'
assert all(p.is_file() for p in (EMA_CHECKPOINT,RAW_CHECKPOINT,TRAIN_METADATA))
print(json.loads(TRAIN_METADATA.read_text(encoding='utf-8'))['best_ema_epoch'],
      json.loads(TRAIN_METADATA.read_text(encoding='utf-8'))['best_raw_epoch'])


In [ ]:
# Build exactly one clean HC3 graph, then evaluate both checkpoint types.
subprocess.run([sys.executable,'building_graph.py','--file',TEST_SPLIT],cwd=REPO,check=True)
GRAPH=REPO/'graph_data'/f'{TEST_SPLIT}.pkl'
assert GRAPH.is_file()

from types import SimpleNamespace
import importlib.util
spec=importlib.util.spec_from_file_location('v3_eval',REPO/'evaluate_v3_quang.py')
v3_eval=importlib.util.module_from_spec(spec); spec.loader.exec_module(v3_eval)
expected=sum(bool(line.strip()) for line in (REPO/'data/original_text/benchmarks/hc3/hc3_test.json').open(encoding='utf-8'))
assert expected == 1000
reports={}; configs=[]
for weight, checkpoint in {'ema':EMA_CHECKPOINT,'raw':RAW_CHECKPOINT}.items():
    # The evaluator expects a canonical name. This copy preserves the original checkpoint.
    stage=EVAL_DIR/'staged'/weight; stage.mkdir(parents=True,exist_ok=True)
    alias=stage/f'v3_baseline_seed{SEED}.pt'; shutil.copy2(checkpoint,alias)
    args=SimpleNamespace(variant='baseline',train_domain='gpt35',checkpoint_dir=str(stage),seeds=[SEED],
                         batch_size=BATCH_SIZE,struct_mode='low',device='cuda',graph_budget_root=str(WORK))
    config=v3_eval.make_config(args); config['weight_source']=weight
    config_path=EVAL_DIR/f'config_{weight}.json'; v3_eval.write_json(config_path,config); configs.append(config_path)
    report=EVAL_DIR/f'hc3_0pct_{weight}.json'
    subprocess.run([sys.executable,'evaluate_v3_quang.py','_test','--config',str(config_path),'--seed',str(SEED),
                    '--graph',str(GRAPH),'--expected-samples',str(expected),
                    '--signature',f'gpt35_to_hc3_0pct_{weight}_independent_best','--report',str(report)],cwd=REPO,check=True)
    reports[weight]=json.loads(report.read_text(encoding='utf-8'))

for weight, report in reports.items():
    print(weight.upper(), {k:report[k] for k in ('accuracy','auc','f1','recall_ai','recall_human','validation_threshold','checkpoint')})

In [ ]:
import zipfile
summary={weight:{k:report[k] for k in ('accuracy','auc','f1','recall_ai','recall_human','threshold','validation_threshold','best_epoch','weight_source') if k in report}
         for weight,report in reports.items()}
(EVAL_DIR/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
archive=WORK/f'v3_dual_ema_raw_gpt35_to_hc3_0pct_seed{SEED}_batch{BATCH_SIZE}.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as out:
    for path in (EMA_CHECKPOINT,RAW_CHECKPOINT,TRAIN_METADATA,DUAL_TRAINER,*configs,
                 EVAL_DIR/'hc3_0pct_ema.json',EVAL_DIR/'hc3_0pct_raw.json',EVAL_DIR/'summary.json'):
        out.write(path,path.name)
print(json.dumps(summary,indent=2))
print('ZIP:',archive)